In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
3# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/sample_submission.csv
/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/all_data.csv
/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/test_public_expanded.csv
/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/test_private_expanded.csv
/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/toxicity_individual_annotations.csv
/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/train.csv
/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/identity_individual_annotations.csv
/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/test.csv


3

In [2]:
#load the training path
training_path='/kaggle/input/competitions/jigsaw-unintended-bias-in-toxicity-classification/train.csv'

#load the training data
df=pd.read_csv(training_path,nrows=200000)

In [3]:
#info of dataset
print("rows and columns:", df.shape)
print("row info:", df.head())
print("target column name:", df.columns)
print("nums of missing value:", df.info())

rows and columns: (200000, 45)
row info:       id    target                                       comment_text  \
0  59848  0.000000  This is so cool. It's like, 'would you want yo...   
1  59849  0.000000  Thank you!! This would make my life a lot less...   
2  59852  0.000000  This is such an urgent design problem; kudos t...   
3  59855  0.000000  Is this something I'll be able to install on m...   
4  59856  0.893617               haha you guys are a bunch of losers.   

   severe_toxicity  obscene  identity_attack   insult  threat  asian  atheist  \
0         0.000000      0.0         0.000000  0.00000     0.0    NaN      NaN   
1         0.000000      0.0         0.000000  0.00000     0.0    NaN      NaN   
2         0.000000      0.0         0.000000  0.00000     0.0    NaN      NaN   
3         0.000000      0.0         0.000000  0.00000     0.0    NaN      NaN   
4         0.021277      0.0         0.021277  0.87234     0.0    0.0      0.0   

   ...  article_id    rating  fun

In [4]:
# get info of target
print("info of target:", df.target.describe())

#create a new row
df["is_toxic"]= (df["target"]>=0.5).astype(int)
print(df[["target", "is_toxic"]].head(20))

#check the nums of toxic comment
print(df["is_toxic"].value_counts())
print(df["is_toxic"].value_counts(normalize=True))

info of target: count    200000.000000
mean          0.098020
std           0.194695
min           0.000000
25%           0.000000
50%           0.000000
75%           0.166667
max           1.000000
Name: target, dtype: float64
      target  is_toxic
0   0.000000         0
1   0.000000         0
2   0.000000         0
3   0.000000         0
4   0.893617         1
5   0.666667         1
6   0.457627         0
7   0.000000         0
8   0.000000         0
9   0.000000         0
10  0.000000         0
11  0.440000         0
12  0.000000         0
13  0.600000         1
14  0.500000         1
15  0.000000         0
16  0.000000         0
17  0.000000         0
18  0.000000         0
19  0.500000         1
is_toxic
0    184622
1     15378
Name: count, dtype: int64
is_toxic
0    0.92311
1    0.07689
Name: proportion, dtype: float64


In [5]:
#get the length of comment
df["comment_length"]=df["comment_text"].str.len()

#get average  length


#get mean and median
print("mean value:",df.groupby("is_toxic")["comment_length"].mean())
print("median value:",df.groupby("is_toxic")["comment_length"].median())

#show some sample
print(df.groupby("is_toxic")[["comment_text", "target", "is_toxic"]].sample(5))

mean value: is_toxic
0    309.797619
1    286.542918
Name: comment_length, dtype: float64
median value: is_toxic
0    209.0
1    201.0
Name: comment_length, dtype: float64
                                             comment_text    target  is_toxic
55367                          Many Alaskans don't think.  0.400000         0
3823    Sounds like this Edwards is a real renegade wh...  0.166667         0
77846   I want a pistol packing, William H. Seward typ...  0.000000         0
106451  Muzzle flashes are short enough that they migh...  0.000000         0
155743  Voting for Caldwell is just a vote for no acco...  0.000000         0
159074  Obstructionist should get voted out of office....  0.500000         1
11203                       This dispute needs more guns!  0.800000         1
95607   Proving once again the Hillary supporters are ...  0.500000         1
23046   The only bad thing is Tallman shot him in the ...  0.500000         1
86170   The dude couldnt even wash dishes and ma

In [6]:

#check the identity cols

print(df.columns)  #list columns name

identity_cols=[
    'asian', 'atheist', 'bisexual',
       'black', 'buddhist', 'christian', 'female', 'heterosexual', 'hindu',
       'homosexual_gay_or_lesbian', 'intellectual_or_learning_disability',
       'jewish', 'latino', 'male', 'muslim', 'other_disability',
       'other_gender', 'other_race_or_ethnicity', 'other_religion',
       'other_sexual_orientation', 'physical_disability',
       'psychiatric_or_mental_illness', 'transgender', 'white'
]   #create identity_cols

print(df[identity_cols].notnull().sum().sort_values(ascending=False))

Index(['id', 'target', 'comment_text', 'severe_toxicity', 'obscene',
       'identity_attack', 'insult', 'threat', 'asian', 'atheist', 'bisexual',
       'black', 'buddhist', 'christian', 'female', 'heterosexual', 'hindu',
       'homosexual_gay_or_lesbian', 'intellectual_or_learning_disability',
       'jewish', 'latino', 'male', 'muslim', 'other_disability',
       'other_gender', 'other_race_or_ethnicity', 'other_religion',
       'other_sexual_orientation', 'physical_disability',
       'psychiatric_or_mental_illness', 'transgender', 'white', 'created_date',
       'publication_id', 'parent_id', 'article_id', 'rating', 'funny', 'wow',
       'sad', 'likes', 'disagree', 'sexual_explicit',
       'identity_annotator_count', 'toxicity_annotator_count', 'is_toxic',
       'comment_length'],
      dtype='object')
asian                                  39723
atheist                                39723
bisexual                               39723
black                                  39

In [7]:
#check sub-type toxic
toxicity_cols = [
    "severe_toxicity",
    "obscene",
    "identity_attack",
    "insult",
    "threat"
]
print("info:",df[toxicity_cols].describe())

#get mean value for subtype

print("Mean value Descending order:", df[df["is_toxic"]==1][toxicity_cols].mean().sort_values(ascending=False))
print((df[df["is_toxic"]==1][toxicity_cols]>=0.5).sum())
print((df[df["is_toxic"]==1][toxicity_cols]>=0.5).info())

info:        severe_toxicity        obscene  identity_attack         insult  \
count    200000.000000  200000.000000    200000.000000  200000.000000   
mean          0.004576       0.015752         0.017246       0.076825   
std           0.022920       0.072224         0.069398       0.172172   
min           0.000000       0.000000         0.000000       0.000000   
25%           0.000000       0.000000         0.000000       0.000000   
50%           0.000000       0.000000         0.000000       0.000000   
75%           0.000000       0.000000         0.000000       0.000000   
max           0.534819       1.000000         1.000000       1.000000   

              threat  
count  200000.000000  
mean        0.010512  
std         0.053688  
min         0.000000  
25%         0.000000  
50%         0.000000  
75%         0.000000  
max         1.000000  
Mean value Descending order: insult             0.551093
obscene            0.136053
identity_attack    0.096993
threat          

In [8]:
#create is_identity and set threshold
print(df["muslim"].describe())
#identity just mentioned instead of null
print((df["muslim"] >= 0.5).sum())
print((df["muslim"]>=0.5).sum()/df["muslim"].count())

count    39723.000000
mean         0.028589
std          0.154989
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: muslim, dtype: float64
1200
0.030209198701004455


In [9]:
print((df[identity_cols] >= 0.5).sum().sort_values(ascending=False))

female                                 4918
male                                   4198
white                                  2304
black                                  1924
christian                              1625
muslim                                 1200
homosexual_gay_or_lesbian               966
psychiatric_or_mental_illness           740
jewish                                  378
transgender                             295
asian                                   266
latino                                  242
atheist                                 113
heterosexual                             98
hindu                                    49
other_race_or_ethnicity                  43
bisexual                                 37
buddhist                                 36
other_religion                           20
physical_disability                      15
intellectual_or_learning_disability      11
other_sexual_orientation                  3
other_disability                

In [10]:
for col in identity_cols:
    mask = df[col]>=0.5
    count= mask.sum()
    rate = df.loc[mask, "is_toxic"].mean()
    print(col, count,rate)

asian 266 0.12030075187969924
atheist 113 0.11504424778761062
bisexual 37 0.08108108108108109
black 1924 0.3024948024948025
buddhist 36 0.08333333333333333
christian 1625 0.15569230769230769
female 4918 0.14599430662871085
heterosexual 98 0.24489795918367346
hindu 49 0.10204081632653061
homosexual_gay_or_lesbian 966 0.2857142857142857
intellectual_or_learning_disability 11 0.0
jewish 378 0.16666666666666666
latino 242 0.2231404958677686
male 4198 0.15340638399237733
muslim 1200 0.28
other_disability 1 0.0
other_gender 1 1.0
other_race_or_ethnicity 43 0.20930232558139536
other_religion 20 0.25
other_sexual_orientation 3 0.0
physical_disability 15 0.2
psychiatric_or_mental_illness 740 0.20270270270270271
transgender 295 0.2305084745762712
white 2304 0.2847222222222222


In [11]:
mask2=df["muslim"].notnull()

rate2 = df.loc[mask2,"is_toxic"].mean()
print("Identity-annotated baseline toxic rate:",rate2) 


Identity-annotated baseline toxic rate: 0.1176900032726632


In [12]:
#create an empty list
results=[]

#accord identity mentioned and toxic data
for col in identity_cols:
    results.append({
        "identity": col,
        "count": (df[col]>=0.5).sum(),
        "toxic_rate":df.loc[df[col]>=0.5, "is_toxic"].mean()
    })
#convert to df
identity_summary=pd.DataFrame(results)

#calculate rate difference
identity_summary["rate_diff"]=(
    identity_summary["toxic_rate"]-rate2
)

#filter-small sample
identity_summary_filtered = identity_summary[
    identity_summary["count"] >= 100
]

#sort value
print(identity_summary_filtered.sort_values(
    "rate_diff",
    ascending=False
))

                         identity  count  toxic_rate  rate_diff
3                           black   1924    0.302495   0.184805
9       homosexual_gay_or_lesbian    966    0.285714   0.168024
23                          white   2304    0.284722   0.167032
14                         muslim   1200    0.280000   0.162310
22                    transgender    295    0.230508   0.112818
12                         latino    242    0.223140   0.105450
21  psychiatric_or_mental_illness    740    0.202703   0.085013
11                         jewish    378    0.166667   0.048977
5                       christian   1625    0.155692   0.038002
13                           male   4198    0.153406   0.035716
6                          female   4918    0.145994   0.028304
0                           asian    266    0.120301   0.002611
1                         atheist    113    0.115044  -0.002646


# Train the model

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

#get x and y col
X=df["comment_text"]
y=df["is_toxic"]

#split training and validation datasets
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
#print(y_train.value_counts(normalize=True),y_valid.value_counts(normalize=True))
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train)
X_valid_tfidf=tfidf.transform(X_valid)

print(X_train_tfidf.shape)
print(X_valid_tfidf.shape)

(160000, 86683)
(40000, 86683)


# Logistic Regression
Firstly, create a reliable model

In [14]:
# import model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

#create a baseline Logistic Regression model
model_1= LogisticRegression(max_iter=1000)

#train th model
model_1.fit(X_train_tfidf,y_train)
y_pred_1=model_1.predict(X_valid_tfidf)

#Evaluate the model_1
accuracy_1=accuracy_score(y_valid,y_pred_1)
class_1=classification_report(y_valid,y_pred_1)
matrix_1=confusion_matrix(y_valid,y_pred_1)
print("accuracy:",accuracy_1,"\n"
"classification:",class_1,"\n"
"confusion matrix:",matrix_1)

accuracy: 0.943175 
classification:               precision    recall  f1-score   support

           0       0.95      0.99      0.97     36924
           1       0.84      0.33      0.47      3076

    accuracy                           0.94     40000
   macro avg       0.89      0.66      0.72     40000
weighted avg       0.94      0.94      0.93     40000
 
confusion matrix: [[36727   197]
 [ 2076  1000]]


# handle class imbalance

In [15]:
#create model_2, JUST change the class_weight
model_2= LogisticRegression(
    max_iter=1000,
class_weight="balanced")

model_2.fit(X_train_tfidf,y_train)
y_pred_2=model_2.predict(X_valid_tfidf)

#Evaluate the model_1
accuracy_2=accuracy_score(y_valid,y_pred_2)
class_2=classification_report(y_valid,y_pred_2)
matrix_2=confusion_matrix(y_valid,y_pred_2)
print("accuracy:",accuracy_2,"\n"
"classification:",class_2,"\n"
"confusion matrix:",matrix_2)

accuracy: 0.906275 
classification:               precision    recall  f1-score   support

           0       0.98      0.92      0.95     36924
           1       0.44      0.77      0.56      3076

    accuracy                           0.91     40000
   macro avg       0.71      0.84      0.75     40000
weighted avg       0.94      0.91      0.92     40000
 
confusion matrix: [[33887  3037]
 [  712  2364]]


# Precision–Recall Trade-off
model_2 is more 'positive'. we improved the importance of "toxic" sample by using **class_weight="balanced"**.
****Precision will be lower than model1, but Recall will be improved****

# Which is BETTER?
For Trust&Safety problem, there is not absolutely correct answer. If we want to ban comments permanently with the model, it will be very dangerous with model_2. However, if we just sent toxic comments to manual audit, model_2 maybe work better.


In [16]:
proba_1=model_1.predict_proba(X_valid_tfidf)
toxic_prob_1=proba_1[:,1]
threshold=0.4
y_pred_threshold=(toxic_prob_1 >= threshold).astype(int)

accuracy_3=accuracy_score(y_valid,y_pred_threshold)
class_3=classification_report(y_valid,y_pred_threshold)
matrix_3=confusion_matrix(y_valid,y_pred_threshold)
print("accuracy:",accuracy_3,"\n"
"classification:",class_3,"\n"
"confusion matrix:",matrix_3)


accuracy: 0.944975 
classification:               precision    recall  f1-score   support

           0       0.95      0.99      0.97     36924
           1       0.78      0.40      0.53      3076

    accuracy                           0.94     40000
   macro avg       0.86      0.69      0.75     40000
weighted avg       0.94      0.94      0.94     40000
 
confusion matrix: [[36570   354]
 [ 1847  1229]]


In [17]:
# small threshold experiment
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds=[0.2, 0.3, 0.4, 0.5, 0.6]

print("Threshold,    Precision,    Recall,        F1","\n")
for threshold in thresholds:
    y_pred_temp=(toxic_prob_1>=threshold).astype(int)
    precision=precision_score(y_valid,y_pred_temp)
    recall=recall_score(y_valid,y_pred_temp)
    f1=f1_score(y_valid,y_pred_temp)
    print(threshold, precision, recall, f1,"\n")



Threshold,    Precision,    Recall,        F1 

0.2 0.5981575603557815 0.6121586475942783 0.6050771208226221 

0.3 0.7069288389513109 0.4908972691807542 0.5794320798158097 

0.4 0.7763739734680986 0.3995448634590377 0.5275810259712385 

0.5 0.835421888053467 0.3250975292587776 0.46805523051720105 

0.6 0.8744725738396625 0.26950585175552666 0.4120278330019881 



# Precision Recall Curve
To get all Precision and Recall of each threshold

In [18]:
from sklearn.metrics import precision_recall_curve

precision_values,recall_values,thresholds=precision_recall_curve(
    y_valid,
    toxic_prob_1
)

#print(len(thresholds))
#print(thresholds[:5])

#calculate f1 values
f1_values=(
    2*precision_values[:-1]*recall_values[:-1]
    /
    (precision_values[:-1]+recall_values[:-1])
)

#find the best f1 value
best_index = np.argmax(f1_values)

best_threshold = thresholds[best_index]
best_fa=f1_values[best_index]
best_precision = precision_values[best_index]
best_recall = recall_values[best_index]

print("Best Threshold:",best_threshold,"\n"
     "Precision:",best_precision,"\n"
     "Recall:",best_recall,"\n"
     "F1:",best_fa,"\n")

Best Threshold: 0.1976244461495725 
Precision: 0.594679186228482 
Recall: 0.6176853055916776 
F1: 0.6059639610907351 



In [19]:
#generate final validation prediction
best_pred=(toxic_prob_1 >=best_threshold).astype(int)

accuracy_4=accuracy_score(y_valid,best_pred)
class_4=classification_report(y_valid,best_pred)
matrix_4=confusion_matrix(y_valid,best_pred)
print("accuracy:",accuracy_4,"\n"
"classification:",class_4,"\n"
"confusion matrix:",matrix_4)

accuracy: 0.938225 
classification:               precision    recall  f1-score   support

           0       0.97      0.96      0.97     36924
           1       0.59      0.62      0.61      3076

    accuracy                           0.94     40000
   macro avg       0.78      0.79      0.79     40000
weighted avg       0.94      0.94      0.94     40000
 
confusion matrix: [[35629  1295]
 [ 1176  1900]]


# Error Analysis

In [26]:
#create a validation results tabel
valid_results=pd.DataFrame({
    "comment_text":X_valid.to_numpy(),
    "actual": y_valid.to_numpy(),
    "predicted": best_pred,
    "probability": toxic_prob_1
    
})

#false_positive
false_positive=valid_results[
    (valid_results["actual"]==0)
    &
    (valid_results["predicted"]==1)
]

#false_negative
false_negative=valid_results[
    (valid_results["actual"]==1)
    &
    (valid_results["predicted"]==0)
]

#print(len(false_positive))
#print(len(false_negative))
pd.set_option("display.max_colwidth", None)

#Random Sampling
print(false_positive[["comment_text", "probability"]].sample(10))
print(false_negative[["comment_text", "probability"]].sample(10))

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 comment_text  \
2761   What I "*realize*" is that Muslims do this because of *their* God, and *their* hate-filled religion.\n\nWhat I "*realize*" is that you cannot honestly be attempting to compare any other faith with Islamist extremists. Yes-  "*some Christians used to terror